# Build & Diagnose a Predictive Organism

**This notebook is the primary entrypoint for the CCR research project.**

It runs the full pipeline end-to-end: record nursery scenarios in Crafter,
quality-gate the data, train a recurrent action-conditioned CNN world model
(the Predictive Cortex), and measure whether it actually learned the world's
dynamics — with explicit pass/fail assertions at every milestone.

### What we're measuring

The core research question: **can a CNN world model, trained incrementally
from recorded experience like an LLM trains on text, learn to predict its own
future senses well enough to beat trivial baselines?**

| Milestone | Claim | Measured by |
|---|---|---|
| **M2a** | Cortex beats copy-last-frame at every prediction horizon | MSE on held-out seeds (step 6) |
| **M2b** | The model actually uses action information | Action-ablation test (step 7) |
| **M4** | Dreams work — hippocampal seeds replay generatively | Dream strip export (step 10) |
| **M5** | Generative replay reduces catastrophic forgetting | Before/after loss comparison (step 11) |

Additional quality checks: frozen-rollout detection (collapsed model), yaw
linear probe (hidden state encodes heading), representation collapse
diagnostics, and reward/terminal/risk/uncertainty head validation.

### Prerequisites

```bash
pip install -e ".[neural]"    # from the repo root
```

### CLI equivalents

Every step below has a CLI equivalent for scripting or CI. The key commands:
- `ccr nursery run all` — record scenarios (step 2)
- `ccr nursery joint` — train + evaluate joint model (steps 4-8)
- `node viewer/server.js --data-dir runs/Pixel` — inspect results (step 12)

## 1. Configuration

All paths and hyperparameters in one place. Change `ORGANISM` to start a
fresh experiment; everything (recordings, checkpoints, exports) namespaces
under it.

In [ ]:
import os, pathlib
import matplotlib.pyplot as plt

# --- Experiment identity ---
ORGANISM = "Pixel"          # names the model id and every file it generates
WORLD = "crafter"           # the deterministic pixel-native nursery world

# --- Paths ---
RECORD_DIR = pathlib.Path("runs") / ORGANISM
RECORD_DIR.mkdir(parents=True, exist_ok=True)

# --- Training hyperparameters ---
HORIZONS_TICKS = (1, 4, 8)  # prediction horizons in ticks
TRAINING_OBJECTIVE = "autoregressive"  # "autoregressive" trains every prefix; "windowed_rollout" is the baseline
EPOCHS = 30
BACKBONE = "gru"             # "gru" (default), "dilated_conv", or "transformer"
EMA_DECAY = 0.99             # Polyak target encoder decay (None to disable)

# --- Scenarios to record and train on ---
SCENARIOS = ["walk_forward", "turn", "approach_entity", "object_permanence"]

print(f"Experiment: {ORGANISM}")
print(f"Recording to: {RECORD_DIR}")
print(f"Horizons: {HORIZONS_TICKS}, Objective: {TRAINING_OBJECTIVE}, Backbone: {BACKBONE}")

## 2. Record nursery data (train + holdout)

Each scenario is a scripted micro-world that isolates one regularity the
model should learn (forward motion, turning, approaching objects, occlusion).
For each scenario, we record 6 train-seed episodes and 2 held-out-seed
episodes in Crafter with pixel frames.

**CLI equivalent:**
```bash
ccr nursery run all --record-dir runs/Pixel --name Pixel
```

In [ ]:
from cognitive_runtime.training.nursery import (
    NurseryConfig, run_nursery_scenario, _scenarios_for_world,
)

cfg = NurseryConfig(world=WORLD, name=ORGANISM, export_predictions=True)
train_dirs, holdout_dirs = [], []
for name in SCENARIOS:
    _model, report = run_nursery_scenario(str(RECORD_DIR), name, cfg)
    train_dirs.extend(report.train_sessions)
    holdout_dirs.extend(report.holdout_sessions)
print("train:", train_dirs)
print("holdout:", holdout_dirs)

## 3. Data-quality gates — halt on red

Before training, every recorded session is checked for data quality: pixel
provenance (are frames real?), motion floor (did the agent actually move?),
episode completeness, and frozen-recording detection. A "red" session means
the data is unusable — we refuse to train on it.

**This gate runs automatically before training in the CLI pipeline too.**

In [ ]:
from cognitive_runtime.record.quality import evaluate_record_quality

for d in train_dirs + holdout_dirs:
    verdict = evaluate_record_quality(d)
    print(f"{verdict.verdict:>5}  {d}  {verdict.issues}")
    assert verdict.verdict != "red", f"refusing to train on a red session: {d}"

## 4. Build the joint action-conditioned dataset

Merge all recorded scenarios into a single dataset of (pixel, action, target)
sequences. The model will learn to predict future frames conditioned on the
action taken — this is why action-ablation (step 7) can test whether it
actually uses that information.

In [ ]:
from cognitive_runtime.training.action_world_model import build_action_sequence_dataset
from cognitive_runtime.programs.crafter.actions import ACTION_SPACE  # pin full vocab

action_keys = [a.name for a in ACTION_SPACE]
train_ds = build_action_sequence_dataset(train_dirs, action_keys=action_keys)
holdout_ds = build_action_sequence_dataset(holdout_dirs, action_keys=action_keys)
print("train transitions:", len(train_ds), "| pixel shape:", train_ds.pixel_shape)

## 5. Train the Predictive Cortex

Train two models on the same data:
- **EMA model** (with Polyak target encoder) — the one we're testing
- **Baseline model** (shared encoder, no EMA) — to confirm EMA helps

The cortex is a recurrent CNN that predicts future pixel frames and latent
states at each horizon. The autoregressive objective trains on every prefix
of the sequence (like an LLM), not just fixed windows.

**CLI equivalent:**
```bash
ccr nursery joint --record-dir runs/Pixel --epochs 30 --backbone gru \
    --training-objective autoregressive --ema-target-decay 0.99
```

In [ ]:
from cognitive_runtime.training.action_world_model import (
    ActionWorldModelConfig, train_action_world_model,
)
from dataclasses import replace

model_cfg = ActionWorldModelConfig(
    horizons_ticks=HORIZONS_TICKS, epochs=EPOCHS, backbone=BACKBONE,
    ema_target_decay=EMA_DECAY, training_objective=TRAINING_OBJECTIVE,
)

# Train the baseline (no EMA) and the EMA model on identical data
baseline_model, baseline_stats = train_action_world_model(
    train_ds, replace(model_cfg, ema_target_decay=None)
)
model, stats = train_action_world_model(train_ds, model_cfg)

# Plot loss curves — should decrease smoothly; a plateau suggests underfitting
fig, ax = plt.subplots(figsize=(8, 4))
for key in ("total_loss", "pixel_loss", "latent_loss"):
    ax.plot(stats["loss_curves"][key], label=key)
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
ax.set_title(f"Cortex training ({TRAINING_OBJECTIVE}, {BACKBONE})")
plt.tight_layout(); plt.show()

## 6. MILESTONE 2a — beat copy-last at every horizon

**The core result.** The trained cortex must predict held-out frames better
than simply copying the last frame (the trivial baseline). We check:

1. **EMA <= baseline MSE** — the Polyak target encoder didn't hurt
2. **model/copy-last ratio < 1.0 at every horizon** — the model adds value
3. **No frozen rollout** — the model hasn't collapsed to a fixed point

If any assertion fails, the model has not learned the world's dynamics and
should not be promoted.

In [ ]:
from cognitive_runtime.training.action_world_model import (
    evaluate_action_world_model, horizons_ticks_to_frames,
)

hf = horizons_ticks_to_frames(HORIZONS_TICKS, holdout_ds.ticks_per_frame)
ev = evaluate_action_world_model(model, holdout_ds, hf)
baseline_ev = evaluate_action_world_model(baseline_model, holdout_ds, hf)
ema_mse = sum(r["model_mse"] for r in ev["horizons"].values()) / len(hf)
baseline_mse = sum(r["model_mse"] for r in baseline_ev["horizons"].values()) / len(hf)
print(f"EMA held-out MSE={ema_mse:.6f}; shared-encoder baseline={baseline_mse:.6f}")
assert ema_mse <= baseline_mse, "EMA target regressed held-out prediction"
for h, row in ev["horizons"].items():
    print(f"t+{h}: model/copy-last={row['model_over_copy_last_mse']:.3f} "
          f"beats={row['beats_copy_last']} PSNR={row['psnr_model']:.1f}dB")
assert ev["rollout_health"]["frozen_rollout"] is False, "cortex collapsed to a fixed point"
assert all(r["beats_copy_last"] for r in ev["horizons"].values()), "Milestone 2a failed"

## 7. MILESTONE 2b — withholding actions must hurt

If the model truly uses action information (not just copying pixels), then
training it *without* actions should produce worse predictions. We train an
identical model with zeroed-out actions and compare held-out MSE.

**Expected result:** with-actions MSE < without-actions MSE.

In [ ]:
from cognitive_runtime.training.nursery import run_action_ablation_eval

abl = run_action_ablation_eval(train_dirs, holdout_dirs, action_keys=action_keys,
                               horizons_ticks=HORIZONS_TICKS)
print(abl)  # expect: with-actions MSE < without-actions MSE (the model uses actions)

## 8. Representation probe — does the hidden state encode heading?

A linear probe on the GRU hidden state: if it can decode the agent's heading
(yaw) where the raw latent cannot, the recurrence is carrying useful motion
state, not just copying pixels.

Also checks for representation collapse — a known failure mode where the
encoder maps everything to the same neighborhood (the V-JEPA pitfall). We
measure effective rank and variance of the latent space.

In [ ]:
from cognitive_runtime.training.action_world_model import representation_collapse_diagnostics

representation = representation_collapse_diagnostics(model, holdout_ds, config=model_cfg)
print(representation)
if representation["gate_evaluable"]:
    assert representation["passed"], "cortex representation collapsed; do not promote"
else:
    print("representation gate N/A: too few yaw-labelled holdout frames")

## 9. Auxiliary heads diagnostic

The cortex also predicts reward, terminal (death), risk, and its own
uncertainty at each horizon. This step checks that each head beats a
constant-predictor baseline (predicting the mean), and that predicted
uncertainty correlates positively with actual prediction error — meaning
the model knows when it doesn't know.

In [ ]:
import torch
from cognitive_runtime.training.action_world_model import _episode_tensors, evaluate_cortex_heads

episodes = _episode_tensors(holdout_ds, model.reconstruction_shape)
heads_report = evaluate_cortex_heads(model, holdout_ds, hf)
for h, row in heads_report.items():
    corr, (ci_lo, ci_hi) = row["uncertainty_error_correlation"], row["uncertainty_error_correlation_ci"]
    print(f"t+{h}: reward beats-const={row['reward_beats_constant']} "
          f"terminal beats-const={row['terminal_beats_constant']} "
          f"risk beats-const={row['risk_beats_constant']} "
          f"uncertainty-error corr={corr:+.3f} (CI {ci_lo:+.3f}..{ci_hi:+.3f})")
    # `beats_constant` is None for a degenerate (zero-variance) target
    # column -- e.g. a holdout split with no death events, where the
    # constant predictor scores an unbeatable exact 0.0 -- so only a
    # literal `False` (the head actually loses to the baseline) fails.
    for head in ("reward", "terminal", "risk"):
        assert row[f"{head}_beats_constant"] is not False, f"t+{h}: {head} head loses to the constant baseline"
    assert ci_lo > 0.0, f"t+{h}: uncertainty/realized-error correlation CI is not clear of 0"

## 10. MILESTONE 4 — dream from a hippocampal seed

A "dream" is: take a stored initial condition (latent state + action
sequence) from the hippocampus and roll the cortex forward with no live
senses. If the dream looks like the actual experience, the world model has
learned enough to imagine plausible futures.

The exported dream strip shows: **seen frame -> predicted frame -> actual
frame -> |error|** at each horizon.

In [ ]:
from brain.hippocampus import Hippocampus, SeedTags
from sleep.dream import dream, export_dream_file

ep, pix, tgt, act = episodes[0]
with torch.no_grad():
    z0 = model.encoder(pix[:1]).squeeze(0)
hippo = Hippocampus(capacity=256)
seed = hippo.encode(z=z0.tolist(),
                    actions=[model.action_keys[int(a)] for a in act[: max(hf)]],
                    tags=SeedTags(dopamine=0.0, threat=0.0, novelty=1.0),
                    source=holdout_dirs[0])
actual_frames = [pix[i] for i in range(max(hf) + 1)]
out_path = export_dream_file(seed, model, actual_frames, horizons=list(hf),
                             session_dir=holdout_dirs[0], name=ORGANISM)
print("dream strip:", out_path)

## 11. MILESTONE 5 — forgetting under generative replay vs flat training

**The sharp research bet.** Train the cortex on scenario A, measure held-out
loss. Then train on scenario B two ways:
- **Flat** — just train on B (catastrophic forgetting expected)
- **Staged + dream replay** — train on B while replaying dreams of A

If replay preserves A's accuracy while flat training destroys it, that's the
result: developmental staging + generative replay reduces catastrophic
forgetting.

See `tests/test_forgetting_metric.py` for the full reference wiring.

In [ ]:
from sleep.forgetting import compute_forgetting_metric
# old_before / old_after are held-out loss samples on scenario A, measured
# before and after training scenario B (populate from evaluate_action_world_model
# per-episode samples under each condition).
old_before = []  # e.g. ev_A['per_episode_model_mse'][hf[0]]
old_after = []   # same, after training B
if old_before and old_after:
    report = compute_forgetting_metric(old_before, old_after,
                                       old_scenario="walk_forward", new_scenario="turn",
                                       tolerance=0.02)
    print("retained:", report.retained, "| forgetting:", report.forgetting_amount)

## 12. Inspect results in the clinic

`export_predictions=True` (set in step 2) wrote prediction files next to each
recorded episode. The clinic viewer shows dream strips, neuromodulator
timelines, and data-quality diagnostics in the browser.

```bash
node viewer/server.js --data-dir runs/Pixel
# open http://localhost:8787
```

Each horizon strip shows: **seen t -> predicted t+h -> actual t+h -> |error|**,
plus the EEG (neuromodulator timelines), attention, and development panels.

## 13. Statistical referee — confidence intervals, not single samples

For any comparison (before/after, candidate/baseline), route through the
statistical evaluation harness so regressions are flagged with confidence
intervals, not judged off a single noisy sample.

**CLI equivalent:**
```bash
ccr statistical-evaluate --policies random,scripted --episodes 20 --baseline random \
    --confidence 0.95
```

## Summary of results

After running all cells, you should have:

| Step | Check | Status |
|---|---|---|
| 3 | All sessions pass data-quality gate | `assert verdict != "red"` |
| 6 | Cortex beats copy-last at every horizon (M2a) | `assert beats_copy_last` |
| 6 | No frozen rollout | `assert frozen_rollout is False` |
| 7 | Action ablation hurts predictions (M2b) | with-actions MSE < without |
| 8 | Representation not collapsed | `assert passed` |
| 9 | Auxiliary heads beat constant baseline | `assert beats_constant is not False` |
| 9 | Uncertainty correlates with error | `assert ci_lo > 0` |
| 10 | Dream strip exported (M4) | file written |
| 11 | Replay reduces forgetting (M5) | `retained == True` |

**Artifacts produced:**
- `runs/Pixel/` — all recorded sessions with pixel frames
- Dream strip files next to each episode
- Prediction exports for the clinic viewer
- Loss curve plots (inline above)

**Next steps:**
- Open the clinic (`node viewer/server.js --data-dir runs/Pixel`) to visually
  inspect dream strips and neuromodulator timelines
- Try different backbones (`BACKBONE = "transformer"`) or objectives
  (`TRAINING_OBJECTIVE = "windowed_rollout"`) and compare
- Run with more epochs or scenarios for a stronger result